In [1]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer

In [2]:
input_file = "/data/elugos/event_embedding/train.csv"

df = pd.read_csv(input_file)

df.columns

/tmp/ipykernel_2528183/431309081.py:3: DtypeWarning: Columns (39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


Index(['GlobalEventID', 'date', 'Year', 'Actor1Code', 'Actor1Name',
       'Actor1CountryCode', 'Actor1EthnicCode', 'Actor2Code', 'Actor2Name',
       'Actor2CountryCode', 'Actor2EthnicCode', 'IsRootEvent', 'EventCode',
       'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale',
       'NumMentions', 'NumSources', 'NumArticles', 'AvgTone',
       'Actor1Geo_CountryCode', 'Actor2Geo_CountryCode', 'ActionGeo_Type',
       'ActionGeo_Fullname', 'ActionGeo_CountryCode', 'ActionGeo_ADM1Code',
       'ActionGeo_Lat', 'ActionGeo_Long', 'ActionGeo_FeatureID', 'DATEADDED',
       'SOURCEURL', 'domain', 'target', 'title', 'text', 'description',
       'sbert_text_title', 'embedding_json_title', 'sbert_text',
       'embedding_json'],
      dtype='object')

In [3]:

import re
import pandas as pd

def extract_first_paragraphs(df: pd.DataFrame, col: str='text', min_len: int=40):
    """
    Extract the first meaningful paragraph from a dataframe column containing article text.

    Heuristics:
      - Prefer chunks separated by blank lines.
      - Skip leading short/byline/date-like chunks.
      - A 'real' paragraph should be >= min_len OR contain sentence-ending punctuation.
      - Fallback: if no blank lines exist, assemble from lines after skipping short/noisy headers.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    col : str
        Column name containing the article text.
    min_len : int
        Minimum length threshold for a chunk to be considered a paragraph.
    
    Returns
    -------
    pandas.Series
        A series containing the extracted first paragraph for each row (or None if not found).
    """

    # Sentence end detection (loose heuristic)
    # sentence_end_re = re.compile(r'[.!?]["\')]*\s|[.!?]["\')]*$')
    sentence_end_re = re.compile(r"(.+)[\n$]")

    # Patterns commonly seen in noisy headers/bylines/dates (non-exhaustive)
    noisy_line_res = [
        re.compile(r'^\s*(?:by|author|reporting|editor|correspondent)\b', re.I),  # byline
        re.compile(r'^\s*(?:updated|published|last\s+modified)\b', re.I),         # metadata
        re.compile(r'^\s*\(?\d{1,2}\s+\w+\s+\d{4}\)?$', re.I),                    # "14 Dec 2025"
        re.compile(r'^\s*\w+\s+\d{1,2},\s+\d{4}\s*$', re.I),                      # "December 14, 2025"
        re.compile(r'^\s*(AP|Reuters|AFP)\b', re.I),                               # wire service tags
        re.compile(r'^\s*(Photo|Image|Credit)\b', re.I),                           # media credits
        re.compile(r'^[A-Z \-]{6,}$'),                                             # all-caps headers
    ]


    def looks_noisy(line: str) -> bool:
        s = line.strip()
        if not s:
            return True
        # Very short lines without sentence punctuation
        if len(s) < min_len and not sentence_end_re.search(s):
            return True
        # Known noisy patterns
        for rx in noisy_line_res:
            if rx.search(s):
                return True
        return False

    def is_paragraph_candidate(chunk: str) -> bool:
        s = chunk.strip()
        if not s:
            return False
        # Consider a chunk a paragraph if it's long OR has sentence-ending punctuation
        return len(s) >= min_len or bool(sentence_end_re.search(s))

    def extract(text):
        if not isinstance(text, str):
            return None

        # Normalize whitespace
        s = text.strip()
        if not s:
            return None

        # First pass: split by blank lines (common paragraph boundary)
        paragraphs = [p.strip() for p in re.split(r'\n\s*\n', s) if p.strip()]

        # Skip initial noisy chunks
        for p in paragraphs:
            if is_paragraph_candidate(p) and not looks_noisy(p):
                return p

        # Fallback: build from individual lines (when no blank lines or all chunks seemed noisy)
        lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
        # Skip leading noisy lines
        i = 0
        while i < len(lines) and looks_noisy(lines[i]):
            i += 1

        if i >= len(lines):
            return None

        # Accumulate lines until we hit an empty separator (rare here since empties were removed)
        # or until we have a candidate paragraph.
        acc = []
        for j in range(i, len(lines)):
            acc.append(lines[j])
            para = ' '.join(acc)
            if is_paragraph_candidate(para):
                return para

        # If nothing matched the heuristics, return the remaining accumulated text (best effort)
        return ' '.join(acc) if acc else None

    return df['text'].apply(extract)


In [4]:
df.columns

Index(['GlobalEventID', 'date', 'Year', 'Actor1Code', 'Actor1Name',
       'Actor1CountryCode', 'Actor1EthnicCode', 'Actor2Code', 'Actor2Name',
       'Actor2CountryCode', 'Actor2EthnicCode', 'IsRootEvent', 'EventCode',
       'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale',
       'NumMentions', 'NumSources', 'NumArticles', 'AvgTone',
       'Actor1Geo_CountryCode', 'Actor2Geo_CountryCode', 'ActionGeo_Type',
       'ActionGeo_Fullname', 'ActionGeo_CountryCode', 'ActionGeo_ADM1Code',
       'ActionGeo_Lat', 'ActionGeo_Long', 'ActionGeo_FeatureID', 'DATEADDED',
       'SOURCEURL', 'domain', 'target', 'title', 'text', 'description',
       'sbert_text_title', 'embedding_json_title', 'sbert_text',
       'embedding_json'],
      dtype='object')

In [5]:
# Extract first paragraphs from input text

first_paras = extract_first_paragraphs(df)
df['first_para'] = first_paras



In [6]:
# First paragraph sanity check
df[['text', 'first_para']]
i=39
print(df['first_para'][i])
print("---")
print(df['text'][i])

“I’m just notifying John Deere right now, if you do that, we’re putting a 200% tariff on everything that you want to sell into the United States,” Trump said Monday, citing reports about the company shifting manufacturing to Mexico.
---
“I’m just notifying John Deere right now, if you do that, we’re putting a 200% tariff on everything that you want to sell into the United States,” Trump said Monday, citing reports about the company shifting manufacturing to Mexico.

Deere earlier this year said it would lay off 503 workers in Illinois and 310 in Iowa as it faces rising operational costs and declining demand. It’s also acquiring land in Mexico to shift some production previously done in the US.

The world’s top farm machinery maker, which also has operations in South America and Europe, is an iconic American firm with its green and yellow tractors. Moline, Illinois-based Deere said in a statement it is committed to US manufacturing with $2 billion invested in domestic plants since 2019.

In [7]:
# Load SBERT
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

/home/jupyter/miniconda3/envs/jupyter-lab/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [8]:
# Extract embeddings
df = df.dropna(subset=['text', 'first_para', 'title'])
df = df.reset_index()
first_para_emb = model.encode(df['first_para'])
df['first_para_emb'] = pd.Series(first_para_emb.tolist())

In [10]:
df.to_csv("/data/elugos/event_embedding/train_20251214.csv", index=None)